# Who Should the Bank Target?
### Bank marketing campaign analysis | Python + Power BI

**Business objective:** Identify customer segments with stronger term-deposit conversion rates and evaluate the efficiency of repeated campaign outreach.

**Data:** UCI Bank Marketing (`bank-full.csv`), 45,211 customer records. The target `y` indicates whether a customer subscribed to a term deposit. This notebook prepares the data and documents the analysis behind the Power BI dashboard.

## 1. Load and inspect the data
Upload `bank-full.csv` when prompted in Google Colab. The original file is semicolon-delimited.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("bank-full.csv", sep=";")

print("Rows and columns:", df.shape)
display(df.head())
print("Missing values:", df.isna().sum().sum())

## 2. Campaign baseline
Create a binary conversion indicator and calculate the overall conversion rate.

In [ ]:
df["converted"] = df["y"].map({"yes": 1, "no": 0})

total_customers = len(df)
total_conversions = int(df["converted"].sum())
overall_rate = df["converted"].mean() * 100
print(f"Customers: {total_customers:,}")
print(f"Conversions: {total_conversions:,}")
print(f"Overall conversion rate: {overall_rate:.2f}%")

**Finding:** Of 45,211 contacted customers, 5,289 subscribed, giving an overall conversion rate of **11.70%**.

## 3. Conversion by occupation
Compare conversion rates by occupation, not just the number of customers in each group.

In [ ]:
job_summary = df.groupby("job").agg(customers=("converted", "size"), conversions=("converted", "sum"), conversion_rate=("converted", "mean"))
job_summary["conversion_rate"] *= 100
job_summary = job_summary.sort_values("conversion_rate", ascending=False)
display(job_summary.round(2))

ax = job_summary["conversion_rate"].plot(kind="bar", figsize=(10, 4))
ax.axhline(overall_rate, linestyle="--", color="gray", label=f"Overall: {overall_rate:.2f}%")
ax.set(title="Term Deposit Conversion Rate by Occupation", xlabel="Occupation", ylabel="Conversion rate (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(); plt.tight_layout(); plt.show()

**Finding:** Students (28.68%) and retirees (22.79%) had the highest occupation-level conversion rates. Blue-collar customers were the largest occupational group but converted at 7.27%. These are associations, not proof that occupation causes conversion.

## 4. Conversion by age group
Create interpretable age bands to compare customers at different life stages.

In [ ]:
age_bins = [17, 29, 39, 49, 59, 69, 100]
age_labels = ["18-29", "30-39", "40-49", "50-59", "60-69", "70+"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)
age_summary = df.groupby("age_group", observed=True).agg(customers=("converted", "size"), conversion_rate=("converted", "mean"))
age_summary["conversion_rate"] *= 100
display(age_summary.round(2))

ax = age_summary["conversion_rate"].plot(kind="bar", figsize=(8, 4))
ax.axhline(overall_rate, linestyle="--", color="gray", label=f"Overall: {overall_rate:.2f}%")
ax.set(title="Term Deposit Conversion Rate by Age Group", xlabel="Age group", ylabel="Conversion rate (%)")
plt.xticks(rotation=0); plt.legend(); plt.tight_layout(); plt.show()

**Finding:** Customers aged 60–69 (29.67%) and 70+ (42.42%) converted at especially high rates; ages 18–29 (17.60%) were also above average. The 70+ group is relatively small (554 customers), so results should be interpreted cautiously.

## 5. Contact frequency and outreach efficiency
Compare conversion with the number of campaign contacts. The `campaign` field is the number of contacts during this campaign.

In [ ]:
contact_bins = [0, 1, 2, 3, 5, float("inf")]
contact_labels = ["1", "2", "3", "4-5", "6+"]
df["contact_frequency"] = pd.cut(df["campaign"], bins=contact_bins, labels=contact_labels)

contact_efficiency = df.groupby("contact_frequency", observed=True).agg(
    customers=("converted", "size"),
    total_contacts=("campaign", "sum"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
)
contact_efficiency["conversion_rate"] *= 100
contact_efficiency["contacts_per_conversion"] = contact_efficiency["total_contacts"] / contact_efficiency["conversions"]
display(contact_efficiency.round(2))

high_frequency = contact_efficiency.loc["6+"]
print(f"6+ share of customers: {high_frequency['customers'] / total_customers * 100:.1f}%")
print(f"6+ share of campaign contacts: {high_frequency['total_contacts'] / df['campaign'].sum() * 100:.1f}%")

**Finding:** Customers contacted once converted at 14.60%, compared with 5.81% for those contacted 6+ times. The 6+ group accounted for about 9.6% of customers but 34.4% of campaign contacts. The latter group averaged approximately 169.7 contacts per conversion, compared with 6.9 for the one-contact group. These group-level measures do **not** establish that repeated calls caused lower conversion.

## 6. Previous campaign outcomes
Compare current conversion rates based on the recorded outcome of a previous marketing campaign.

In [ ]:
previous_summary = df.groupby("poutcome").agg(customers=("converted", "size"), conversion_rate=("converted", "mean"))
previous_summary["conversion_rate"] *= 100
display(previous_summary.sort_values("conversion_rate", ascending=False).round(2))

**Finding:** Customers with a recorded previous campaign success converted at 64.73%, compared with 11.70% overall. However, previous outcome was `unknown` for most customers, limiting how broadly this signal can be used.

## 7. Business recommendations

1. **Prioritize proven responders:** Give greater consideration to customers with recorded prior campaign success when that information is available.
2. **Test higher-converting segments:** Evaluate targeted campaigns for younger and older age groups, students, and retirees, while validating results on future campaigns.
3. **Reassess repeated outreach:** Review customers after several unsuccessful contacts and test whether redirecting contact capacity improves efficiency.

**Limitations:** These descriptive comparisons do not demonstrate causation. Age and occupation overlap with other customer characteristics. Most prior campaign outcomes are unknown. Call duration should not be used to choose whom to contact because it is only known after a call.

## 8. Export for Power BI
The exported CSV contains the original data plus `converted`, `age_group`, and `contact_frequency`.

In [ ]:
output_file = "bank_marketing_cleaned.csv"
df.to_csv(output_file, index=False)
files.download(output_file)